# LangGraph: an explicit graph

A graph makes an agent's control flow visible as nodes and edges.

In this notebook, we will:

- run a prebuilt tool agent;
- define graph state;
- create model, tool, and refund nodes;
- route with conditional edges; and
- pause before a risky node for approval.

Before running this notebook, install the graph extra once with: pip install -e ".[langgraph]".

In [ ]:
import os
import re
from getpass import getpass
from pathlib import Path
from typing import Annotated, TypedDict

import litellm
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
TEMPERATURE = 0
litellm.drop_params = True

## 1. Create the model and tools

LangGraph uses LangChain model and tool objects. The underlying shop data is the same.

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from shoplab import world

orders = {item["order_id"]: item for item in world.load_orders()}
customers = {item["customer_id"]: item for item in world.load_customers()}

provider_model = MODEL.split("/", 1)[1] if MODEL.startswith("openrouter/") else MODEL
model = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    model=provider_model,
    temperature=TEMPERATURE,
)

@tool
def get_order(order_id: str) -> dict:
    """Look up one Larkspur order."""
    return orders.get(order_id, {"error": "order not found"})

@tool
def get_customer(customer_id: str) -> dict:
    """Look up one Larkspur customer."""
    return customers.get(customer_id, {"error": "customer not found"})

@tool
def find_policy(query: str) -> list:
    """Search Larkspur policy documents."""
    return world.search_policy(query)

READ_TOOLS = [get_order, get_customer, find_policy]
print([item.name for item in READ_TOOLS])

## 2. Use the prebuilt graph

create_react_agent packages a model node, a tool node, and the loop between them.

In [ ]:
from langgraph.prebuilt import create_react_agent

SYSTEM = (
    "You handle Larkspur returns. Look up the order, customer, and policy. "
    "End with: DECISION: decision | policy_id | amount."
)
TICKET = (
    "Ticket TKT-2205: order ORD-7312, customer CUST-07. "
    "Opened Torrent boots were returned after 18 days for a refund."
)

prebuilt = create_react_agent(model, READ_TOOLS, prompt=SYSTEM)
prebuilt_result = prebuilt.invoke({
    "messages": [{"role": "user", "content": TICKET}]
})

print("compiled type:", type(prebuilt).__name__)
print("final:", prebuilt_result["messages"][-1].content)

## 3. Build the graph ourselves

Graph state is the data passed from node to node. Our state contains messages. The add_messages reducer appends new messages instead of replacing the list.

In [ ]:
from langgraph.graph import END, START, StateGraph, add_messages
from langgraph.prebuilt import ToolNode
from shoplab.tools import Ledger

class TriageState(TypedDict):
    messages: Annotated[list, add_messages]

model_with_tools = model.bind_tools(READ_TOOLS)
ledger = Ledger()

def triage_node(state: TriageState):
    reply = model_with_tools.invoke(state["messages"])
    return {"messages": [reply]}

def refund_node(state: TriageState):
    text = state["messages"][-1].content or ""
    amounts = re.findall(r"\d+\.\d{2}", text)
    amount = float(amounts[-1]) if amounts else 0.0
    entry = ledger.record("issue_refund", order_id="ORD-7312", amount_usd=amount)
    return {"messages": [{"role": "assistant", "content": f"refund recorded: {entry}"}]}

The router reads the latest state and chooses the next edge.

In [ ]:
def choose_next(state: TriageState):
    last = state["messages"][-1]

    if getattr(last, "tool_calls", None):
        return "tools"

    text = (last.content or "").lower()
    if "approve_refund" in text or "partial_refund" in text:
        return "refund"

    return END

In [ ]:
def graph_builder():
    graph = StateGraph(TriageState)
    graph.add_node("triage", triage_node)
    graph.add_node("tools", ToolNode(READ_TOOLS))
    graph.add_node("refund", refund_node)

    graph.add_edge(START, "triage")
    graph.add_conditional_edges("triage", choose_next, ["tools", "refund", END])
    graph.add_edge("tools", "triage")
    graph.add_edge("refund", END)
    return graph

seed = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": TICKET},
]

ungated = graph_builder().compile()
for edge in ungated.get_graph().edges:
    print(edge.source, "->", edge.target)

## 4. Pause before the risky node

interrupt_before stops the graph before refund runs. A checkpointer keeps the state while a human reviews it.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

ledger.entries.clear()
checkpointer = MemorySaver()
gated = graph_builder().compile(
    checkpointer=checkpointer,
    interrupt_before=["refund"],
)

config = {"configurable": {"thread_id": "TKT-2205"}}
paused = gated.invoke({"messages": seed}, config)
snapshot = gated.get_state(config)

print("next node:", snapshot.next)
print("ledger entries:", ledger.entries)
print("proposal:", paused["messages"][-1].content)

No refund was recorded. After approval, invoking with None resumes from the saved state.

In [ ]:
finished = gated.invoke(None, config)

print("final message:", finished["messages"][-1].content)
print("ledger entries:", ledger.entries)
print("next node:", gated.get_state(config).next)

The graph needs more setup than a small while loop. In return, the path, state, pause point, and resume behavior are first-class parts of the system.

## What to remember

1. LangGraph makes control flow explicit.
2. State moves between nodes.
3. Edges describe fixed movement.
4. Conditional edges act as routers.
5. A checkpointer preserves paused state.
6. interrupt_before creates a clear approval point.

## Small practice steps

1. Print the graph edges in a different order.
2. Add an escalation node.
3. Inspect snapshot.values before resuming.